
# 📈 NeMo Safe Synthesizer Tutorial: Time Series

#### What you'll learn

In this notebook, we'll explore how to use NeMo Safe Synthesizer for **time-series data**: loading a time-series classification dataset, configuring the synthesizer for temporal data, generating synthetic time series, and visualizing the synthetic version of the time series data.

A full run takes about 20 minutes on an A100. If you have not yet completed the [Safe Synthesizer 101](safe-synthesizer-101.ipynb) tutorial, consider starting there first.

### 🖥️ Prerequisites

This notebook requires a Linux machine with an NVIDIA GPU (H100 recommended, A100 minimum) and CUDA 12.9+. It will not run on macOS, Windows, or Apple Silicon.

### ⚡ Install Safe Synthesizer

Run the cell below to install NeMo Safe Synthesizer (engine and CUDA 12.9) and the `aeon` toolkit for loading the dataset and running TSTR evaluation.

In [ ]:
%%bash
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

if command -v uv > /dev/null 2>&1; then
    uv pip install "nemo-safe-synthesizer[engine,cu129]" --index https://flashinfer.ai/whl/cu129 --index https://download.pytorch.org/whl/cu129 --index https://wheels.vllm.ai/88d34c6409e9fb3c7b8ca0c04756f061d2099eb1/cu129 --index-strategy unsafe-best-match
    uv pip install aeon matplotlib
else
    pip install "nemo-safe-synthesizer[engine,cu129]" --extra-index-url https://flashinfer.ai/whl/cu129 --extra-index-url https://download.pytorch.org/whl/cu129 --extra-index-url https://wheels.vllm.ai/88d34c6409e9fb3c7b8ca0c04756f061d2099eb1/cu129
    pip install aeon matplotlib
fi


### 📥 Load and preview the ECG200 dataset

We use the [ECG200](https://www.timeseriesclassification.com/description.php?Dataset=ECG200) (R. Olszewski and the UCR/TSML Archive) dataset from the UCR time-series archive. Each sample is a single-lead ECG recording (96 timesteps) classified as normal or abnormal heartbeat.

| Detail | Value |
|--------|-------|
| Features/channels | 1 ECG amplitude feature (univariate) |
| Sequence length | 96 timesteps |
| Number of sequences (train / test) | 100 / 100 |
| Classes | 2 (-1: normal, 1: abnormal) |

We load the dataset with [aeon](https://www.aeon-toolkit.org/) and convert it to the long-format DataFrame that Safe Synthesizer expects. Each time series becomes a group of rows identified by `group_id`, with a `timestep` column for the time index.

> Note: This demo uses a univariate time series. Safe Synthesizer also supports multivariate time series with numeric, categorical, and text features.

> Note: Data disclaimer: Each user is responsible for checking dataset content and applicable licenses, then determining whether the dataset is suitable for the intended use.

In [ ]:
import warnings

import numpy as np
import pandas as pd
from aeon.datasets import load_classification


def aeon_to_long_df(X, y):
    """Convert aeon arrays (n_samples, n_channels, length) to NSS long-format DataFrame."""
    n_samples, _, length = X.shape
    rows = []
    for i in range(n_samples):
        for timestep in range(length):
            rows.append({"timestep": timestep, "ecg": X[i, 0, timestep], "label": int(y[i]), "group_id": i})
    return pd.DataFrame(rows)


load_kwargs = {"load_equal_length": True, "load_no_missing": True}
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="Call to deprecated function .*load_classification.*",
        category=FutureWarning,
    )
    X_train_raw, y_train = load_classification("ECG200", split="train", **load_kwargs)
    X_test_raw, y_test = load_classification("ECG200", split="test", **load_kwargs)

train_df = aeon_to_long_df(X_train_raw, y_train)
test_df = aeon_to_long_df(X_test_raw, y_test)

print(f"Train: {X_train_raw.shape[0]} series × {X_train_raw.shape[2]} timesteps = {len(train_df)} rows")
print(f"Test:  {X_test_raw.shape[0]} series × {X_test_raw.shape[2]} timesteps = {len(test_df)} rows")
train_df.head(10)

### 🔍 Visualize real ECG traces

Before generating synthetic data, let’s look at a few real ECG recordings from each class.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
class_labels = {-1: "Normal (-1)", 1: "Abnormal (1)"}

for ax, cls in zip(axes, [-1, 1]):
    group_ids = train_df[train_df["label"] == cls]["group_id"].unique()[:5]
    for gid in group_ids:
        series = train_df[train_df["group_id"] == gid]
        ax.plot(series["timestep"].values, series["ecg"].values, alpha=0.6)
    ax.set_title(f"Class: {class_labels[cls]}")
    ax.set_xlabel("Timestep")

axes[0].set_ylabel("ECG amplitude")
fig.suptitle("Real ECG200 traces (5 per class)", fontsize=13)
plt.tight_layout()
plt.show()

### ⚙️ Configure and run Safe Synthesizer

Create the Safe Synthesizer builder and configure it for time-series synthesis:

- `with_time_series` enables time-series mode and specifies the timestamp column.
- `with_data` tells the synthesizer that rows sharing the same `group_id` belong to a single time series. We set `holdout=0` so the tutorial trains on all 100 training sequences; TSTR evaluation uses the original ECG200 test split instead.
- `with_replace_pii(enable=False)` disables PII replacement because this numeric ECG dataset does not contain PII columns.
- `with_train` sets demo-specific training hyperparameters.
- `with_generate` enables timestamp fidelity enforcement. `num_records` is not applicable for time series mode as generation will generate all groups.

We skip the built-in evaluation step because time-series-specific quality metrics (TSTR, similarity scores, etc.) are under active development and will be available in a future release. Instead, we demonstrate TSTR evaluation manually in a later section.

Refer to the [configuration docs](../user-guide/configuration.md) for the full list of options.

In [ ]:
from nemo_safe_synthesizer.sdk.library_builder import SafeSynthesizer

builder = (
    SafeSynthesizer()
    .with_data_source(train_df)
    .with_time_series(
        is_timeseries=True,
        timestamp_column="timestep",
    )
    .with_data(
        group_training_examples_by="group_id",
        max_sequences_per_example=None,
        holdout=0,
    )
    .with_replace_pii(enable=False)
    .with_train(
        # These settings were selected for this small ECG200 demo.
        num_input_records_to_sample=144000,
        learning_rate=5e-4,
        lora_r=32,
    )
    .with_generate(
        enforce_timeseries_fidelity=True,
    )
    .with_evaluate(enabled=False)
)
builder.run()
results = builder.results

### 📤 Retrieve synthetic data

Inspect the generated synthetic data including row count and preview of the first rows.

In [ ]:
synth = results.synthetic_data
n_synth_groups = synth["group_id"].nunique()
print(f"Generated {n_synth_groups} synthetic time series ({len(synth)} rows)")
synth.head(10)

### 🔬 Visual comparison: Real vs. Synthetic

Let’s compare real and synthetic ECG traces side by side for each class.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True)

for col, (source_name, source_df) in enumerate([("Real", train_df), ("Synthetic", synth)]):
    for row, cls in enumerate([-1, 1]):
        ax = axes[row, col]
        group_ids = source_df[source_df["label"] == cls]["group_id"].unique()[:8]
        for gid in group_ids:
            series = source_df[source_df["group_id"] == gid]
            ax.plot(series["timestep"].values, series["ecg"].values, alpha=0.6)
        ax.set_title(f"{source_name} — {class_labels[cls]}")
        ax.set_xlabel("Timestep")
    axes[0, col].set_ylabel("ECG amplitude")
    axes[1, col].set_ylabel("ECG amplitude")

fig.suptitle("Real vs. Synthetic ECG200 traces (8 per class)", fontsize=13)
plt.tight_layout()
plt.show()

### 🎯 Evaluate with TSTR (Train on Synthetic, Test on Real)

[TSTR](https://arxiv.org/abs/1706.02633) is the standard way to measure whether synthetic data preserves the discriminative patterns of the original. The idea is simple:

1. Baseline: Train a classifier on *real* data, evaluate on the held-out *real* test set.
2. TSTR: Train the same classifier on *synthetic* data, evaluate on the *real* test set.

If the synthetic data captures the class-relevant temporal structure, the TSTR accuracy should be close to the baseline. We use [MiniRocket](https://arxiv.org/abs/2012.08791), a fast and accurate time-series classifier, for this comparison.

The first three timesteps are used as prefill/context records during generation, so they are not part of the generated signal we want to score. We drop those prefill rows before TSTR training and apply the same trim to the real test split so train and test sequences have matching lengths.

> Note: Built-in time-series evaluation metrics—including automated TSTR—are under active development and will be integrated into the Safe Synthesizer pipeline in a future release.

In [ ]:
from aeon.transformations.collection.convolution_based import MiniRocket
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline


def long_df_to_aeon(df, value_col="ecg"):
    """Convert NSS long-format DataFrame back to aeon (n_samples, n_channels, length) arrays."""
    groups = df.groupby("group_id")
    X_list, y_list = [], []
    for _, grp in groups:
        grp = grp.sort_values("timestep")
        X_list.append(grp[value_col].values)
        y_list.append(grp["label"].iloc[0])
    X = np.array(X_list)[:, np.newaxis, :]  # (n_samples, 1, length)
    y = np.array(y_list, dtype=str)
    return X, y


prefill_timesteps = 3
train_eval_df = train_df[train_df["timestep"] >= prefill_timesteps]
test_eval_df = test_df[test_df["timestep"] >= prefill_timesteps]
synth_eval_df = synth[synth["timestep"] >= prefill_timesteps]

X_train_real, y_train_real = long_df_to_aeon(train_eval_df)
X_test_eval, y_test_eval = long_df_to_aeon(test_eval_df)
X_train_synth, y_train_synth = long_df_to_aeon(synth_eval_df)


# Baseline: train on REAL, test on real
pipe_real = make_pipeline(MiniRocket(), RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
pipe_real.fit(X_train_real, y_train_real)
real_acc = pipe_real.score(X_test_eval, y_test_eval)

# TSTR: train on SYNTHETIC, test on real
pipe_synth = make_pipeline(MiniRocket(), RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
pipe_synth.fit(X_train_synth, y_train_synth)
synth_acc = pipe_synth.score(X_test_eval, y_test_eval)

print(f"Real-trained accuracy:      {real_acc:.3f}")
print(f"Synthetic-trained accuracy:  {synth_acc:.3f}")
print(f"TSTR gap:                    {real_acc - synth_acc:+.3f}")

### 🚀 Interpreting results and next steps

The synthetic-trained classifier will usually trail the real-trained baseline, especially in a small demo dataset like ECG200. A smaller TSTR gap suggests that the synthetic data preserved more class-relevant temporal structure; a larger gap is a signal to inspect the synthetic traces, class balance, and generated sequence lengths.

To reduce the gap on your own dataset, try more training examples and tune generation quality with settings such as `learning_rate`, `lora_r`, and `num_input_records_to_sample`. See the [configuration docs](../user-guide/configuration.md) for the full parameter reference.

Built-in time-series evaluation metrics are still under development. Until those are available, TSTR and visual inspection are useful manual checks for this workflow.